In [1]:
import os
import sys

import numpy as  np
from lwm_multi_model1 import multi_modal_lwm  # 클래스 import
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time

from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

In [8]:
scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

param_manager.params["scenes"] = list(range(100))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

# bandwidth 변경
param_manager.params["comm"]["OFDM"]["bandwidth"] = 0.1   # 예시: 0.1 GHz = 100 MHz

param_manager.params["comm"]["activate_RX_filter"] = 1
param_manager.params["comm"]["generate_OFDM_channels"] = 1
param_manager.params["comm"]["enable_Doppler"] = 1
param_manager.params["comm"]["num_paths"] = 25

print(param_manager.params["comm"]["OFDM"])

dataset = Dataset(param_manager)

{'bandwidth': 0.1, 'subcarriers': 512, 'selected_subcarriers': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]}
Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.79s)
Generating radar dataset: ⏳ In progress


Generating radar dataset: ✅ Completed (186.44s)


In [12]:
comm=dataset.comm_dataset
comm.data[0][0]

{'bs_loc': array([-57.771 ,  96.141 ,   3.4537], dtype=float32),
 'ue': [<deepverse.wireless.Channel.OFDMChannel at 0x7f2c00e6d370>],
 'ue_loc': array([[-44.6921,  94.3239,   1.68  ]], dtype=float32),
 'bs': [<deepverse.wireless.Channel.OFDMChannel at 0x7f2c00e6ed20>]}

In [13]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

tmp = flatten_comm_frames(comm)

In [29]:
tmp[0]["ue"][0].coeffs[0][0][:5]

array([-7.56513577e-07-1.21836854e-06j, -7.85388916e-07-1.18918454e-06j,
       -8.30579254e-07-1.18849282e-06j, -9.15286515e-07-1.18986463e-06j,
       -1.02811517e-06-1.15782456e-06j])

In [15]:
print("bandwidth =", param_manager.params["comm"]["OFDM"]["bandwidth"])

bandwidth = 0.1
